# AI-102 Final Revision Cheat Sheet  
## Scenario → Azure AI Service → Configuration → Code/JSON Patterns

**Purpose:** This notebook is designed for last-4-days AI-102 exam preparation.  
It focuses on what Microsoft-style questions usually test:

1. Which Azure AI service fits a requirement?
2. Which configuration is most appropriate?
3. Which API/SDK pattern or JSON structure matches the scenario?
4. What are the common exam traps?

> Exam mindset: AI-102 is not asking you to invent AI from scratch.  
> It is asking whether you can design and implement Azure AI solutions using the right managed services, security, configuration, and responsible AI controls.

---

## Current exam context

Microsoft's AI-102 study guide says the exam covers these broad areas:

- Plan and manage an Azure AI solution
- Implement generative AI solutions
- Implement agentic solutions
- Implement computer vision solutions
- Implement natural language processing solutions
- Implement knowledge mining and information extraction solutions

Microsoft also notes that AI-102 retires on **June 30, 2026, 11:59 PM Central Standard Time**.

Official reference:  
https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ai-102

# 0. Ultra-fast exam decoding table

| Requirement phrase in question | Think first | Typical Azure service |
|---|---|---|
| "Answer only from internal documents", "citations", "grounded answers" | RAG | Azure OpenAI / Foundry + Azure AI Search |
| "Index documents", "OCR during indexing", "entity extraction", "key phrases", "skillset" | Knowledge mining | Azure AI Search |
| "Search by meaning", "similarity search", "embeddings", "vectors" | Vector search | Azure AI Search vector index + embeddings |
| "Extract fields from invoices/receipts/IDs/tax forms" | Prebuilt extraction | Azure AI Document Intelligence |
| "Extract fields from custom forms" | Custom extraction | Document Intelligence custom model |
| "Different document types mixed together" | Classify then extract / composed model | Document Intelligence classifier/composed model |
| "Detect intent and entities in user utterances" | Bot understanding | Azure AI Language - Conversational Language Understanding |
| "Extract key phrases, named entities, PII, sentiment" | Text analytics | Azure AI Language |
| "Translate text" | Translation | Azure AI Translator |
| "Speech to text", "transcribe calls", "speaker diarization" | Audio transcription | Azure AI Speech |
| "Text to speech", "voice output", "neural voice" | Speech synthesis | Azure AI Speech |
| "OCR from image", "read printed/handwritten text" | Image OCR | Azure AI Vision Read/OCR |
| "Describe image", "tags", "caption", "objects" | Image analysis | Azure AI Vision |
| "Train custom image classifier/detector with labelled images" | Custom image model | Custom Vision / Azure ML AutoML vision depending wording |
| "Face detection/verification/identification" | Face | Azure AI Face, subject to responsible use |
| "Moderate hate/sexual/violence/self-harm content" | Safety moderation | Azure AI Content Safety |
| "Prevent public chatbot unsafe output" | Input + output filtering | Content Safety + system prompt + app controls |
| "Agent calls tools/actions" | Agentic tool use | Azure AI Foundry Agent Service / tool calling |
| "High impact action: payments, claims, legal, medical" | Human/rule guardrails | Agent recommends, deterministic rule/human approves |
| "Use Entra ID instead of keys" | Secure auth | Managed identity + Azure RBAC |
| "Secrets" | Secure storage | Azure Key Vault |
| "Network isolation" | Private access | Private endpoints / VNet / disable public network |

# 1. Plan and manage Azure AI solutions

## 1.1 Resource selection

| Scenario | Best choice | Why |
|---|---|---|
| Need several classic Azure AI services under one endpoint/key | Azure AI multi-service resource / Foundry Tools | Centralized access to many AI services |
| Need GPT/chat/embeddings | Azure OpenAI / Azure AI Foundry Models | Foundation model deployment |
| Need no-code/low-code testing of prompts/models | Azure AI Foundry portal/playground | Experiment, evaluate, deploy |
| Need custom ML training pipeline | Azure Machine Learning | Custom ML lifecycle, training, MLOps |
| Need AI service from app securely | Managed identity + RBAC | Avoid hard-coded keys |
| Need store secrets/config | Azure Key Vault | Secret lifecycle and access control |
| Need production isolation | Private endpoint/VNet | Restrict public access |

## 1.2 Exam traps

| Trap option | Why wrong |
|---|---|
| "Train a custom model for everything" | Usually overkill. Use prebuilt Azure AI services unless custom training is required. |
| "Use one AI resource for OpenAI + Search + Document Intelligence together" | Azure AI Search is its own search service. Azure OpenAI/Foundry model deployments are separate from Search. |
| "Use keys in client-side code" | Bad security. Use backend + managed identity/Key Vault. |
| "Prompt alone prevents hallucination" | Weak. Use grounding/RAG/evaluation/content filters. |
| "LLM directly updates production records" | Unsafe for high-impact actions. Use rule checks/human approval. |

## 1.3 Recommended secure architecture pattern

```text
Client app
   ↓
Backend API / Azure Functions / App Service
   ↓ uses managed identity
Azure AI services / Azure AI Search / Storage / Key Vault
```

Do **not** expose Azure AI keys directly to browsers/mobile apps.

In [ ]:
# Colab setup cell: not required for reading, but useful if you want to test snippets.
# Run only the packages you need.

# !pip install openai azure-identity azure-search-documents azure-ai-documentintelligence azure-ai-textanalytics azure-cognitiveservices-speech azure-ai-vision-imageanalysis azure-ai-contentsafety

# 2. Generative AI / Azure OpenAI / Azure AI Foundry

## 2.1 Scenario mapping

| Requirement | Best pattern |
|---|---|
| General chatbot | Chat completion / responses style API |
| Chatbot must answer from company docs | RAG: retrieve chunks from Azure AI Search, pass as context |
| Need citations | Store source metadata in index; include source references in prompt/output |
| Need semantic similarity | Embeddings + vector search |
| Need search + keyword + vector | Hybrid search |
| Need better ranking over text results | Semantic ranker |
| Need safe public chatbot | Content filters + Content Safety + system prompt + app controls |
| Need evaluate prompt quality | Prompt flow/evaluation in Foundry |
| Need model to call APIs/tools | Tool/function calling or Agent Service |
| Need deterministic business action | LLM proposes; rules/human approves |

## 2.2 RAG decision pattern

Use RAG when:
- Source knowledge changes frequently.
- Answers must be grounded in documents.
- Citations are required.
- The model should not answer from memory.

Avoid relying only on fine-tuning when:
- You need exact factual answers from documents.
- The knowledge changes often.
- You need citations.

Fine-tuning is better for:
- Style, format, tone, domain-specific phrasing.
- Repeated response patterns.
- Not for "memorize all policy documents exactly".

## 2.3 Basic Azure OpenAI chat pattern - Python

```python
from openai import AzureOpenAI
import os

client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version="2024-10-21"  # Use version required by your resource/docs
)

response = client.chat.completions.create(
    model="gpt-4o-mini",  # In Azure this is your deployment name
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Answer concisely."},
        {"role": "user", "content": "Explain Azure AI Search in one paragraph."}
    ],
    temperature=0.2,
    max_tokens=500
)

print(response.choices[0].message.content)
```

Exam detail:
- In Azure OpenAI, `model` often refers to the **deployment name**, not merely the base model name.
- Lower `temperature` is better for factual/business answers.
- Use grounding context for factual enterprise answers.

## 2.4 RAG prompt template

```text
System:
You are an assistant for internal company policy questions.
Answer only using the provided sources.
If the answer is not in the sources, say you do not know.
Cite the source document and section for every factual claim.

User question:
{question}

Retrieved sources:
[Source 1]
Document: {doc_name}
Section: {section}
Content: {chunk_text}

[Source 2]
Document: {doc_name}
Section: {section}
Content: {chunk_text}
```

## 2.5 RAG answer JSON shape

```json
{
  "answer": "Employees are entitled to 20 days of parental leave if they meet the eligibility criteria.",
  "citations": [
    {
      "document": "HR-Parental-Leave-Policy.pdf",
      "section": "Section 4.2",
      "chunk_id": "hr-parental-0042"
    }
  ],
  "confidence": "medium",
  "not_answered_from_memory": true
}
```

Exam idea:
- The citation is not magic. Your app must retrieve source metadata from the search index and instruct the model to include it.

## 2.6 Tool/function calling pattern

Use when the model needs to decide **which function/API** to call.

Example tools:
- get_claim_status(claimId)
- get_policy_coverage(policyId)
- draft_email(customerId, message)
- create_ticket(summary)

For risky operations:
- approve_claim()
- transfer_money()
- update_medical_record()

Use guardrails:
1. Authenticate user.
2. Validate authorization.
3. Run deterministic business rules.
4. Require human approval if high-impact.
5. Log the action.
6. Then execute update.

Bad exam answer:
> Let the model update records directly because confidence is high.

Good exam answer:
> Let the model recommend or prepare action; rules/human approval performs final update.

# 3. Azure AI Search / Knowledge Mining / RAG Indexing

Microsoft describes Azure AI Search as supporting indexing, enrichment, chunking, vector generation, and search over structured content. Vector search supports similarity over numeric vector representations.

Useful official docs:
- Azure AI Search overview: https://learn.microsoft.com/en-us/azure/search/search-what-is-azure-search
- Search index overview: https://learn.microsoft.com/en-us/azure/search/search-what-is-an-index
- Vector search overview: https://learn.microsoft.com/en-us/azure/search/vector-search-overview

## 3.1 Core concepts

| Component | Meaning |
|---|---|
| Data source | Where raw content comes from, e.g. Blob Storage, SQL |
| Index | Searchable schema/store, like a search-optimized table |
| Indexer | Pulls data from source into index |
| Skillset | AI enrichment pipeline during indexing |
| Skill | OCR, entity recognition, key phrase extraction, translation, embedding, etc. |
| Analyzer | Controls tokenization/language processing for text |
| Suggester | Typeahead/autocomplete support |
| Semantic ranker | Improves relevance using semantic understanding |
| Vector field | Stores embeddings |
| Hybrid search | Combines keyword/BM25 and vector search |
| Filterable field | Used in filters/facets |
| Searchable field | Full-text search |
| Retrievable field | Returned in results |
| Sortable field | Used for order by |
| Facetable field | Used for faceted navigation |

## 3.2 Index field attributes - exam table

| Requirement | Set this |
|---|---|
| User searches field text | `searchable: true` |
| App displays field in results | `retrievable: true` |
| Filter by department/category/date | `filterable: true` |
| Sort by date/price/rating | `sortable: true` |
| Facet by category/author | `facetable: true` |
| Field is document key | `key: true` |
| Store vector embedding | vector field with dimensions/profile |
| Store metadata for citations | retrievable metadata fields |

Common trap:
- A field can be present in the index but not returned if `retrievable` is false.
- A field cannot be filtered unless `filterable` is true.

## 3.3 Azure AI Search index JSON - keyword + vector + citations

```json
{
  "name": "policy-index",
  "fields": [
    {
      "name": "id",
      "type": "Edm.String",
      "key": true,
      "filterable": true
    },
    {
      "name": "content",
      "type": "Edm.String",
      "searchable": true,
      "retrievable": true
    },
    {
      "name": "title",
      "type": "Edm.String",
      "searchable": true,
      "filterable": true,
      "retrievable": true
    },
    {
      "name": "sourceFile",
      "type": "Edm.String",
      "filterable": true,
      "facetable": true,
      "retrievable": true
    },
    {
      "name": "section",
      "type": "Edm.String",
      "filterable": true,
      "retrievable": true
    },
    {
      "name": "lastUpdated",
      "type": "Edm.DateTimeOffset",
      "filterable": true,
      "sortable": true,
      "retrievable": true
    },
    {
      "name": "contentVector",
      "type": "Collection(Edm.Single)",
      "searchable": true,
      "retrievable": false,
      "dimensions": 1536,
      "vectorSearchProfile": "vector-profile"
    }
  ],
  "vectorSearch": {
    "algorithms": [
      {
        "name": "hnsw-config",
        "kind": "hnsw"
      }
    ],
    "profiles": [
      {
        "name": "vector-profile",
        "algorithm": "hnsw-config"
      }
    ]
  },
  "semantic": {
    "configurations": [
      {
        "name": "semantic-config",
        "prioritizedFields": {
          "titleField": {
            "fieldName": "title"
          },
          "prioritizedContentFields": [
            {
              "fieldName": "content"
            }
          ],
          "prioritizedKeywordsFields": [
            {
              "fieldName": "sourceFile"
            }
          ]
        }
      }
    ]
  }
}
```

Exam notes:
- Vector field dimensions must match the embedding model.
- Keep vectors usually not retrievable.
- Keep source metadata retrievable for citations.

## 3.4 Azure AI Search Python - create/search pattern

```python
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
import os

search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name="policy-index",
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"])
)

results = search_client.search(
    search_text="parental leave eligibility",
    select=["title", "content", "sourceFile", "section"],
    top=5
)

for r in results:
    print(r["title"], r["sourceFile"], r["section"])
    print(r["content"][:300])
```

## 3.5 Filter query example

```python
results = search_client.search(
    search_text="expense reimbursement",
    filter="department eq 'HR' and lastUpdated ge 2025-01-01T00:00:00Z",
    select=["title", "content", "sourceFile", "section"],
    top=5
)
```

## 3.6 Hybrid/vector search pattern - conceptual

```python
# Pseudocode shape: actual SDK version may vary.
results = search_client.search(
    search_text="Can contractors access learning benefits?",
    vector_queries=[
        {
            "kind": "vector",
            "vector": query_embedding,
            "fields": "contentVector",
            "k": 5
        }
    ],
    select=["title", "content", "sourceFile", "section"],
    top=5
)
```

Exam idea:
- Keyword search is good for exact terms.
- Vector search is good for semantic meaning.
- Hybrid search combines both.
- Semantic ranker improves ranking/captions/answers over text results.

## 3.7 Knowledge mining skillset JSON - OCR + entities + key phrases

```json
{
  "name": "legal-doc-skillset",
  "description": "Extract OCR text, entities, and key phrases",
  "skills": [
    {
      "@odata.type": "#Microsoft.Skills.Vision.OcrSkill",
      "name": "ocr-skill",
      "context": "/document/normalized_images/*",
      "defaultLanguageCode": "en",
      "inputs": [
        {
          "name": "image",
          "source": "/document/normalized_images/*"
        }
      ],
      "outputs": [
        {
          "name": "text",
          "targetName": "ocrText"
        }
      ]
    },
    {
      "@odata.type": "#Microsoft.Skills.Text.EntityRecognitionSkill",
      "name": "entity-skill",
      "context": "/document",
      "categories": ["Person", "Organization", "Location", "DateTime"],
      "defaultLanguageCode": "en",
      "inputs": [
        {
          "name": "text",
          "source": "/document/content"
        }
      ],
      "outputs": [
        {
          "name": "persons",
          "targetName": "people"
        },
        {
          "name": "organizations",
          "targetName": "organizations"
        },
        {
          "name": "locations",
          "targetName": "locations"
        }
      ]
    },
    {
      "@odata.type": "#Microsoft.Skills.Text.KeyPhraseExtractionSkill",
      "name": "keyphrase-skill",
      "context": "/document",
      "defaultLanguageCode": "en",
      "inputs": [
        {
          "name": "text",
          "source": "/document/content"
        }
      ],
      "outputs": [
        {
          "name": "keyPhrases",
          "targetName": "keyPhrases"
        }
      ]
    }
  ]
}
```

Exam pattern:
- "Enrich during indexing" = skillset.
- "Read scanned PDFs/images" = OCR skill.
- "Extract entities/key phrases as metadata" = Language skills in skillset.

# 4. Azure AI Document Intelligence

Official docs:
- Model overview: https://learn.microsoft.com/en-us/azure/ai-services/document-intelligence/model-overview
- Custom models: https://learn.microsoft.com/en-us/azure/ai-services/document-intelligence/train/custom-model
- Custom neural model: https://learn.microsoft.com/en-us/azure/ai-services/document-intelligence/train/custom-neural

## 4.1 Scenario mapping

| Requirement | Use |
|---|---|
| Extract invoice fields | Prebuilt invoice model |
| Extract receipt fields | Prebuilt receipt model |
| Extract ID info | Prebuilt ID document model |
| Extract W-2/tax-like fields if supported | Prebuilt tax model |
| Extract text/tables/layout generally | Layout model |
| Extract from fixed-layout forms | Custom template model |
| Extract from varied/semi-structured documents | Custom neural model |
| Mixed document types | Document classifier and/or composed model |
| Need only OCR from image/PDF | Read/Layout or Vision OCR depending context |
| Need search over extracted docs | Document Intelligence + Azure AI Search |

## 4.2 Model choice traps

| Scenario | Good answer | Bad answer |
|---|---|---|
| Many supplier invoices, no labels | Prebuilt invoice | Train custom from scratch |
| Company-specific form with labels available | Custom model | Prebuilt invoice unless it is invoice |
| Several known custom form types | Composed model / classifier | One generic model without classification |
| Need tables/selection marks/layout | Layout model or custom extraction | OCR only |
| Need just full-text extraction for search | Layout/OCR + Search | Custom extraction model unnecessarily |

## 4.3 Python - prebuilt invoice

```python
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.core.credentials import AzureKeyCredential
import os

endpoint = os.environ["DOCUMENT_INTELLIGENCE_ENDPOINT"]
key = os.environ["DOCUMENT_INTELLIGENCE_KEY"]

client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

with open("invoice.pdf", "rb") as f:
    poller = client.begin_analyze_document(
        model_id="prebuilt-invoice",
        body=f
    )

result = poller.result()

for doc in result.documents:
    fields = doc.fields
    vendor = fields.get("VendorName")
    invoice_id = fields.get("InvoiceId")
    total = fields.get("InvoiceTotal")
    print("Vendor:", vendor.content if vendor else None)
    print("Invoice:", invoice_id.content if invoice_id else None)
    print("Total:", total.content if total else None)
```

## 4.4 Python - layout model

```python
poller = client.begin_analyze_document(
    model_id="prebuilt-layout",
    body=open("document.pdf", "rb")
)
result = poller.result()

for page in result.pages:
    print("Page:", page.page_number)
    for line in page.lines:
        print(line.content)

for table in result.tables:
    for cell in table.cells:
        print(cell.row_index, cell.column_index, cell.content)
```

## 4.5 Custom model training - conceptual steps

1. Put sample documents in Azure Blob Storage.
2. Label fields using Document Intelligence Studio.
3. Train custom model.
4. Test accuracy.
5. Use model ID in SDK/API.
6. For multiple models, compose or classify.

Exam memory:
- Custom extraction requires labelled examples.
- Microsoft docs say custom extraction can get started with as few as five examples of the same form/document type.

# 5. Azure AI Language / NLP

## 5.1 Scenario mapping

| Requirement | Feature |
|---|---|
| Detect user goal from utterance | Conversational Language Understanding (CLU) |
| Extract account number/location/date from utterance | CLU entities |
| Build FAQ question answering over known Q&A | Custom question answering |
| Detect sentiment/opinion | Sentiment analysis/opinion mining |
| Extract names/orgs/locations/dates | Named entity recognition |
| Detect sensitive data | PII detection |
| Extract key phrases | Key phrase extraction |
| Summarize text | Language summarization / generative summarization depending wording |
| Translate text | Azure AI Translator, not Language |
| Convert speech to text | Azure AI Speech, not Language |

## 5.2 CLU design example

Utterances:
- "Report an outage at 2150"
- "My power is off in Blacktown"
- "Show my latest bill"
- "Change my direct debit account"

Intents:
- ReportOutage
- ViewBill
- ChangePaymentMethod
- AskBillingQuestion

Entities:
- postcode
- accountNumber
- outageLocation
- paymentMethod
- date

Exam trap:
- If question asks for **intent + entities in conversation**, choose CLU.
- If question asks for **predefined FAQ answers**, choose Custom Question Answering.

## 5.3 Python - Text Analytics examples

```python
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential
import os

client = TextAnalyticsClient(
    endpoint=os.environ["LANGUAGE_ENDPOINT"],
    credential=AzureKeyCredential(os.environ["LANGUAGE_KEY"])
)

documents = [
    "Contoso Ltd signed the contract with Fabrikam in Sydney on 12 June 2026."
]

entities = client.recognize_entities(documents)
for result in entities:
    for ent in result.entities:
        print(ent.text, ent.category, ent.confidence_score)

key_phrases = client.extract_key_phrases(documents)
for result in key_phrases:
    print(result.key_phrases)

sentiment = client.analyze_sentiment(["The agent was helpful but the wait time was terrible."])
for result in sentiment:
    print(result.sentiment, result.confidence_scores)
```

## 5.4 PII detection

```python
docs = ["My name is Ravi Kumar and my phone number is 0400 123 456."]
result = client.recognize_pii_entities(docs)

for doc in result:
    print("Redacted:", doc.redacted_text)
    for ent in doc.entities:
        print(ent.text, ent.category)
```

# 6. Azure AI Speech

## 6.1 Scenario mapping

| Requirement | Use |
|---|---|
| Real-time speech-to-text | Speech recognition |
| Batch transcription of audio files | Batch transcription |
| Multi-speaker meeting/call transcription | Conversation transcription / diarization |
| Identify who spoke which words | Speaker diarization / conversation transcription |
| Text-to-speech | Speech synthesis |
| Custom acoustic/language adaptation | Custom Speech |
| Translate spoken language | Speech translation |
| Wake word | Keyword recognition |
| Analyze text sentiment after call | Speech → text, then Language or OpenAI |

## 6.2 Exam traps

| Trap | Correction |
|---|---|
| Use Language service for audio | Speech first, then Language for text analytics |
| Use Document Intelligence for recorded calls | Wrong; calls are audio |
| Use OpenAI directly for raw audio transcription in Azure AI-102 classic service questions | Usually Speech service unless question explicitly says model supports audio |
| Need customer vs agent transcript | Conversation transcription / diarization |

## 6.3 Python - speech-to-text from microphone

```python
import azure.cognitiveservices.speech as speechsdk
import os

speech_config = speechsdk.SpeechConfig(
    subscription=os.environ["SPEECH_KEY"],
    region=os.environ["SPEECH_REGION"]
)

speech_config.speech_recognition_language = "en-AU"
audio_config = speechsdk.audio.AudioConfig(use_default_microphone=True)

recognizer = speechsdk.SpeechRecognizer(
    speech_config=speech_config,
    audio_config=audio_config
)

result = recognizer.recognize_once_async().get()

if result.reason == speechsdk.ResultReason.RecognizedSpeech:
    print(result.text)
else:
    print(result.reason)
```

## 6.4 Python - text-to-speech

```python
speech_config = speechsdk.SpeechConfig(
    subscription=os.environ["SPEECH_KEY"],
    region=os.environ["SPEECH_REGION"]
)

speech_config.speech_synthesis_voice_name = "en-AU-NatashaNeural"

synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config)
result = synthesizer.speak_text_async("Hello, this is an Azure AI Speech example.").get()
```

# 7. Computer Vision / Image Analysis / OCR / Custom Vision

## 7.1 Scenario mapping

| Requirement | Service/feature |
|---|---|
| Read text from image | Azure AI Vision OCR/Read |
| Detect objects/tags/caption image | Azure AI Vision Image Analysis |
| Moderate image content | Azure AI Content Safety |
| Train custom classifier/detector with labelled images | Custom Vision or Azure ML vision depending wording |
| Detect product category in labelled shelf images | Custom object detection |
| Detect face attributes/verify identity | Azure AI Face, responsible use constraints |
| Extract fields from scanned forms | Document Intelligence, not generic Vision |
| Search images by text/embedding | Vision + embeddings/search depending scenario |

## 7.2 Vision vs Document Intelligence

| Requirement | Choose |
|---|---|
| Image caption/tags/objects | Vision |
| OCR from natural image | Vision OCR |
| Structured form/invoice/receipt extraction | Document Intelligence |
| Tables/selection marks/layout in PDFs | Document Intelligence Layout |
| Custom product detection from photos | Custom Vision / Azure ML vision |

## 7.3 Python - Image Analysis conceptual

```python
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential
import os

client = ImageAnalysisClient(
    endpoint=os.environ["VISION_ENDPOINT"],
    credential=AzureKeyCredential(os.environ["VISION_KEY"])
)

with open("shelf.jpg", "rb") as image_data:
    result = client.analyze(
        image_data=image_data,
        visual_features=[
            VisualFeatures.CAPTION,
            VisualFeatures.TAGS,
            VisualFeatures.OBJECTS,
            VisualFeatures.READ
        ],
        language="en"
    )

if result.caption:
    print("Caption:", result.caption.text)

if result.read:
    for block in result.read.blocks:
        for line in block.lines:
            print(line.text)
```

Exam:
- `READ`/OCR detects text.
- `TAGS`/`OBJECTS`/`CAPTION` describe visual content.

# 8. Azure AI Content Safety / Responsible AI

## 8.1 Scenario mapping

| Requirement | Use |
|---|---|
| Detect hate/sexual/violence/self-harm | Azure AI Content Safety |
| Check user prompt before model | Prompt/input moderation |
| Check model answer before display | Output moderation |
| Public chatbot safety | Content Safety + Azure OpenAI filters + app controls |
| Block based on severity | Severity threshold routing |
| Human review | Store/reroute borderline or high-risk content |
| Prompt injection detection | Prompt shields / application-level controls depending service wording |
| Enterprise compliance | Logging, monitoring, least privilege, evaluation |

## 8.2 Safety architecture

```text
User input
  ↓
Input moderation / prompt shield
  ↓ if allowed
RAG retrieval / tools
  ↓
LLM generation
  ↓
Output moderation
  ↓
Display answer or block/escalate
```

## 8.3 Safety decision table

| Content severity | Action |
|---|---|
| Safe/low | Allow |
| Medium | Warn, redact, or route depending policy |
| High | Block or escalate |
| Self-harm imminent | Provide crisis-safe response/escalate per policy |
| Prompt injection | Ignore malicious instruction; use trusted system/developer context |

## 8.4 Python - Content Safety conceptual

```python
from azure.ai.contentsafety import ContentSafetyClient
from azure.core.credentials import AzureKeyCredential
from azure.ai.contentsafety.models import AnalyzeTextOptions
import os

client = ContentSafetyClient(
    endpoint=os.environ["CONTENT_SAFETY_ENDPOINT"],
    credential=AzureKeyCredential(os.environ["CONTENT_SAFETY_KEY"])
)

request = AnalyzeTextOptions(text="User supplied text here")
response = client.analyze_text(request)

for category in response.categories_analysis:
    print(category.category, category.severity)

# App logic:
# if severity >= threshold: block/escalate
# else: allow to model
```

Exam trap:
- A system prompt alone is not enough for public/untrusted content.

# 9. Agentic AI / Azure AI Foundry Agent Service / Tool Use

## 9.1 Scenario mapping

| Requirement | Design |
|---|---|
| Agent answers questions from docs | Agent + knowledge/RAG |
| Agent calls internal APIs | Tool/function actions |
| Agent drafts email | Low-risk tool/action |
| Agent updates CRM | Require auth, validation, audit |
| Agent approves claim/payment | Human/rule approval before update |
| Agent escalates complex cases | Escalation tool |
| Agent must run in private network | Private networking/private endpoints |
| Agent needs secure access to resources | Managed identity/RBAC |

## 9.2 High-impact action pattern

```text
LLM/agent:
  - classify request
  - gather evidence
  - produce recommendation

Deterministic service:
  - validate policy/rules
  - check permissions
  - check thresholds
  - check fraud/risk flags

Human:
  - approve if required

System of record:
  - update only after checks
  - log audit trail
```

## 9.3 Tool schema example

```json
{
  "type": "function",
  "function": {
    "name": "get_claim_status",
    "description": "Retrieve claim status by claim ID.",
    "parameters": {
      "type": "object",
      "properties": {
        "claimId": {
          "type": "string",
          "description": "The insurance claim ID"
        }
      },
      "required": ["claimId"]
    }
  }
}
```

## 9.4 Risky tool example - how to guard

```json
{
  "tool": "approve_claim",
  "risk": "high",
  "required_controls": [
    "authenticated_user",
    "authorization_check",
    "amount_under_threshold",
    "policy_active",
    "no_fraud_flags",
    "human_approval_or_rule_engine_approval",
    "audit_log"
  ]
}
```

Exam rule:
- Agents may call tools, but risky external side effects need guardrails.

# 10. Security, identity, networking, and operations

Official docs mention Azure RBAC support for Azure OpenAI and managed identities for Entra-based access. Private networking/VNet/private endpoints are used to restrict access to services.

## 10.1 Authentication options

| Scenario | Best answer |
|---|---|
| Quick dev/test | API key |
| Production backend service | Managed identity + RBAC |
| Avoid storing secrets | Managed identity |
| Need store service keys/secrets | Azure Key Vault |
| Need user-specific access | Microsoft Entra ID |
| Need least privilege | Azure RBAC role assignment |
| Browser/mobile app calls AI | Use backend proxy; do not expose key |

## 10.2 Managed identity pattern

```python
from azure.identity import DefaultAzureCredential
from openai import AzureOpenAI
import os

credential = DefaultAzureCredential()

# Some SDKs support token providers for Entra auth.
# Conceptual pattern:
# 1. Enable managed identity on compute.
# 2. Assign RBAC role on target Azure AI resource.
# 3. Use DefaultAzureCredential in app.
```

## 10.3 Key Vault pattern

```python
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient
import os

credential = DefaultAzureCredential()
vault_url = os.environ["KEY_VAULT_URL"]

secret_client = SecretClient(
    vault_url=vault_url,
    credential=credential
)

search_key = secret_client.get_secret("azure-search-admin-key").value
```

## 10.4 Network isolation

| Requirement | Use |
|---|---|
| Only allow internal network | VNet integration/private endpoint |
| Disable internet exposure | Disable public network access |
| Secure Storage/Search/OpenAI together | Private endpoints + trusted services/managed identity |
| Enterprise bank/regulated workload | Private networking, RBAC, audit logs, monitoring |

Exam trap:
- "Use keys in app config" may be okay for simple examples, but production/security questions usually prefer managed identity + Key Vault/RBAC.

# 11. End-to-end architectures you must recognize

## 11.1 Enterprise HR policy assistant

Requirements:
- Answer from HR docs only
- Citations
- Reduce hallucination
- Secure access

Architecture:
```text
HR documents in Blob/SharePoint
   ↓ indexer
Azure AI Search index + skillset/chunking/embeddings
   ↓ retrieve top chunks
Azure OpenAI/Foundry model
   ↓
Answer with citations
```

Best services:
- Azure AI Search
- Azure OpenAI / Foundry Models
- Managed identity/RBAC
- Content Safety if public or broad audience

Wrong answers:
- Fine-tune only
- Prompt only
- Upload all PDFs into prompt every time

## 11.2 Invoice processing

Requirements:
- Extract vendor, invoice ID, date, total, tax, line items
- Many suppliers
- Fastest implementation

Architecture:
```text
Invoice PDFs
   ↓
Document Intelligence prebuilt-invoice
   ↓
Validation/business rules
   ↓
Store extracted fields
   ↓
Human review for low confidence
```

Use confidence scores to route review:
```json
{
  "field": "InvoiceTotal",
  "value": "$1,245.00",
  "confidence": 0.71,
  "action": "human_review"
}
```

Wrong answers:
- Custom Vision
- Sentiment analysis
- Train from scratch when prebuilt exists

## 11.3 Call center analytics

Requirements:
- Live support calls
- Speech to text
- Speaker separation
- Later summarization

Architecture:
```text
Audio stream
   ↓
Azure AI Speech conversation transcription
   ↓
Transcript with speaker labels
   ↓
Azure AI Language/OpenAI for sentiment/summary
   ↓
CRM/ticket insight
```

Wrong answer:
- Language directly on audio
- Vision OCR

## 11.4 Legal document knowledge mining

Requirements:
- 200,000 docs
- PDFs, Word, scanned images
- OCR
- Entities/key phrases searchable
- Filtering by metadata

Architecture:
```text
Blob Storage
   ↓
Azure AI Search data source
   ↓
Indexer + skillset
   ↓ OCR/entities/key phrases/chunking
Search index
   ↓
User search/filter/RAG
```

Key objects:
- Data source
- Index
- Indexer
- Skillset
- Knowledge store if asked to project enriched data

## 11.5 Public chatbot

Requirements:
- External users
- Unsafe prompts possible
- Unsafe outputs possible
- Need moderation

Architecture:
```text
User prompt
   ↓
Content Safety / prompt shield
   ↓
RAG/tools/model
   ↓
Output moderation
   ↓
Return/block/escalate
```

Wrong answer:
- "Just tell the model to be safe"

# 12. Service-selection mega table

| Service | Choose when | Do not choose when |
|---|---|---|
| Azure OpenAI / Foundry Models | Generate, summarize, reason, chat, embeddings | Need deterministic extraction from invoices without LLM |
| Azure AI Search | Search/index/retrieve/enrich/vector/hybrid/RAG | Need speech transcription |
| Document Intelligence | Extract fields/layout/tables from documents/forms | Need analyze normal photos |
| Azure AI Language | Intent/entity/sentiment/NER/PII/key phrases/Q&A | Need process raw audio |
| Azure AI Speech | Speech-to-text/text-to-speech/speaker diarization | Need analyze text documents |
| Azure AI Vision | OCR/image caption/tags/objects | Need structured invoice fields |
| Custom Vision | Custom labelled image classification/detection | Need document field extraction |
| Content Safety | Moderate unsafe text/images/prompts/outputs | Need search relevance |
| Translator | Translate text/documents | Need intent detection |
| Azure Machine Learning | Custom ML training/MLOps | Prebuilt service can solve requirement |
| Key Vault | Store secrets/keys/certs | Directly generate AI output |
| Managed Identity | Secure service-to-service auth | Local quick test with no Azure resource identity |

# 13. Common "best answer" patterns

## Pattern 1: Prebuilt first

If Microsoft gives a common document type:
- invoice
- receipt
- ID
- tax form
- business card

Pick **prebuilt Document Intelligence** before custom.

## Pattern 2: Custom only when needed

Pick custom when:
- unsupported document/image type
- company-specific fields
- labelled training data exists
- prebuilt accuracy insufficient

## Pattern 3: RAG for factual enterprise knowledge

Pick RAG when:
- internal documents
- citations
- current policies
- reduce hallucinations
- answer only from sources

## Pattern 4: Search skillset for enrichment

Pick skillset when:
- OCR during indexing
- key phrases/entities during indexing
- metadata enrichment
- knowledge mining pipeline

## Pattern 5: Managed identity for production

Pick managed identity/RBAC when:
- secure production app
- avoid keys
- Azure service-to-service auth
- enterprise compliance

## Pattern 6: Human/rule guardrails for actions

Pick rule/human approval when:
- financial impact
- legal impact
- medical impact
- official records
- irreversible action

# 14. Practice-style mini questions

Use these for rapid self-testing.

## Q1
A chatbot must answer only from internal PDFs and include citations.  
**Answer:** Azure OpenAI/Foundry + Azure AI Search RAG.

## Q2
Extract vendor, date, total, tax, and line items from supplier invoices without labels.  
**Answer:** Document Intelligence prebuilt invoice.

## Q3
Detect intent "ReportOutage" and entity "postcode" from user utterance.  
**Answer:** Azure AI Language - Conversational Language Understanding.

## Q4
Transcribe customer-agent calls and identify who spoke.  
**Answer:** Azure AI Speech conversation transcription/diarization.

## Q5
Index scanned legal PDFs and extract organizations/dates during indexing.  
**Answer:** Azure AI Search indexer + skillset with OCR/entity extraction.

## Q6
Public chatbot must block hate/self-harm/violent outputs.  
**Answer:** Azure AI Content Safety plus model/app controls.

## Q7
Detect custom products on shelves using thousands of labelled images.  
**Answer:** Custom Vision/custom object detection.

## Q8
Need filter search results by department.  
**Answer:** Make department field `filterable: true`.

## Q9
Need display document source for citations.  
**Answer:** Store source file/section as `retrievable: true` fields in Search index.

## Q10
Backend app should call Azure AI service without storing keys.  
**Answer:** Managed identity + Azure RBAC.

# 15. Final 4-day exam survival plan

## Day 1
- Read this notebook once.
- Take one practice exam.
- Categorize every wrong answer:
  - Search
  - Document Intelligence
  - OpenAI/RAG
  - Agentic AI
  - Speech
  - Vision
  - Language
  - Safety
  - Security

## Day 2
- Study only your weakest 3 categories.
- Memorize service mapping tables.
- Practice configuration questions.

## Day 3
- Take second full practice exam.
- For every wrong answer, write the "trigger phrase" you missed.
  Example:
  - "filter by category" → field must be `filterable`
  - "citations" → source metadata retrievable + RAG
  - "scanned docs during indexing" → OCR skillset

## Day 4
- Review wrong-answer notebook.
- No deep new study.
- Revise:
  - Search fields
  - Document Intelligence model choice
  - RAG architecture
  - Content Safety
  - Managed identity/RBAC
  - Agent guardrails

## Exam mindset

When stuck, ask:

1. Is this about documents, images, speech, text, search, or generation?
2. Is there a prebuilt service?
3. Does the scenario require custom training?
4. Does it require grounding/citations?
5. Does it require secure production access?
6. Does it change business records or have financial/legal impact?

That usually reveals the answer.

# 16. One-page final memory sheet

```text
Documents/forms/invoices        → Document Intelligence
Search/index/RAG retrieval      → Azure AI Search
Grounded chatbot/citations      → Azure OpenAI + AI Search
Intent/entities in utterance    → CLU
Sentiment/NER/PII/key phrases   → Azure AI Language
Speech-to-text/TTS/calls        → Azure AI Speech
Image OCR/caption/tags/objects  → Azure AI Vision
Custom labelled images          → Custom Vision / Azure ML vision
Unsafe content moderation       → Azure AI Content Safety
Translation                     → Azure AI Translator
Custom ML/MLOps                 → Azure Machine Learning
Secrets                         → Key Vault
Secure service auth             → Managed identity + RBAC
Private enterprise network      → Private endpoints/VNet
High-impact agent action        → rules/human approval
```

Final encouragement:
You do not need perfect Azure mastery to clear AI-102.  
You need strong pattern recognition, enough configuration detail, and disciplined practice review.

# 17. Official references used

- AI-102 study guide: https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ai-102
- Azure AI Search overview: https://learn.microsoft.com/en-us/azure/search/search-what-is-azure-search
- Azure AI Search index overview: https://learn.microsoft.com/en-us/azure/search/search-what-is-an-index
- Azure AI Search vector overview: https://learn.microsoft.com/en-us/azure/search/vector-search-overview
- Document Intelligence model overview: https://learn.microsoft.com/en-us/azure/ai-services/document-intelligence/model-overview
- Document Intelligence custom models: https://learn.microsoft.com/en-us/azure/ai-services/document-intelligence/train/custom-model
- Azure OpenAI RBAC: https://learn.microsoft.com/en-us/azure/foundry-classic/openai/how-to/role-based-access-control
- Azure OpenAI managed identity: https://learn.microsoft.com/en-us/azure/foundry-classic/openai/how-to/managed-identity
- Azure AI services virtual networks/private endpoints: https://learn.microsoft.com/en-us/azure/ai-services/cognitive-services-virtual-networks